# 01 — Trích xuất khung xương MV-Fall v2 (Dataset2) trên Colab

Notebook này trích khung xương 3 luồng (joint / motion / angle) từ **Dataset2**, chỉ lấy 3 người
**Giáp, Minh, Toàn** (bỏ Phong). Kết quả là `custom_all_v2.npz` + `videos_v2.csv` trên Google Drive, dùng cho
`02_train_protocols.ipynb`.

So với `extract_keypoint.py` cũ:

* **Giữ nguyên**: MediaPipe Pose (`model_complexity=1`, `smooth_landmarks=False`, `CONF_THRESH=0.1`),
  14 khớp (13 + Mid-Hip ảo), nội suy frame thiếu, cửa sổ 60 frame không chồng lấn, tọa độ pixel thô.
* **Thêm metadata để chia dữ liệu không rò rỉ**: `subject` (S1–S3), `view` (cam1/cam2), `video_id`, `group_id`,
  `window_idx`, `nonfall_type` (hard/adl/unspecified), `activity`, `orientation` (landscape/portrait).
* **Không tạo video lật**: `is_flip = 0` cho mọi mẫu. Lật ngang chỉ làm online trong tập train (`aug_hflip`).
* **Có cache từng video trên Drive**: Colab ngắt giữa chừng thì chạy lại sẽ tiếp tục, không trích lại từ đầu.

**Chuẩn bị trên Google Drive** (một lần):

```
MyDrive/FallDetection/
└── Dataset2/
    ├── Giáp/  (Fall/..., Non_Fall/...)
    ├── Minh/  (lying_fall, sitting_fall, standing_fall, non_fall)
    └── Toàn/  (lying_fall, sitting_fall, standing_fall, non_fall)
```

Có thể upload cả thư mục Phong, notebook sẽ tự bỏ qua. Nên upload bằng *Google Drive for desktop*
(kéo thả thư mục), không cần nén.

**Runtime**: CPU là đủ (MediaPipe chạy CPU). Chạy lần lượt từ trên xuống.

## 1. Mount Google Drive và cấu hình đường dẫn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT   = '/content/drive/MyDrive/FallDetection'
DRIVE_DATA   = os.path.join(DRIVE_ROOT, 'Dataset2')          # dữ liệu gốc trên Drive
LOCAL_DATA   = '/content/Dataset2'                            # bản sao cục bộ (đọc nhanh hơn Drive)
CACHE_DIR    = os.path.join(DRIVE_ROOT, 'kp_cache_v2')        # cache khung xương từng video
OUT_NPZ      = os.path.join(DRIVE_ROOT, 'custom_all_v2.npz')
OUT_CSV      = os.path.join(DRIVE_ROOT, 'videos_v2.csv')
SANITY_DIR   = os.path.join(DRIVE_ROOT, 'sanity_v2')

# Thư mục người được dùng -> mã ẩn danh. Khóa so khớp sau khi bỏ dấu + chữ thường.
SUBJECT_MAP  = {'giap': 'S1', 'minh': 'S2', 'toan': 'S3'}     # Phong không có ở đây -> bị bỏ qua

for d in [CACHE_DIR, SANITY_DIR]:
    os.makedirs(d, exist_ok=True)
assert os.path.isdir(DRIVE_DATA), f'Không thấy {DRIVE_DATA} — kiểm tra lại đường dẫn upload'
print(sorted(os.listdir(DRIVE_DATA)))

## 2. Cài MediaPipe

MediaPipe bản mới đã bỏ API `mp.solutions.pose` mà code dùng, nên ghim phiên bản `0.10.21`.
Gói này có thể hạ cấp `numpy`; nếu cell báo cần khởi động lại, chọn **Runtime → Restart session**
rồi chạy lại từ **cell 1**, bỏ qua cell này.

Ghi lại phiên bản in ra ở cell kế tiếp để đưa vào mục Implementation details của bài báo.

In [ ]:
import sys
print('Python', sys.version)
!pip -q install "mediapipe==0.10.21"

In [ ]:
import cv2, mediapipe as mp, numpy as np
print('opencv', cv2.__version__, '| mediapipe', mp.__version__, '| numpy', np.__version__)
assert hasattr(mp, 'solutions') and hasattr(mp.solutions, 'pose'), 'Phiên bản mediapipe này không có mp.solutions.pose'

## 3. Sao chép dữ liệu từ Drive về ổ cục bộ của Colab

In [ ]:
import shutil, unicodedata, time

def norm_key(s: str) -> str:
    # 'Giáp' -> 'giap', 'Hoạt động dễ nhầm với ngã' -> 'hoat dong de nham voi nga'
    s = unicodedata.normalize('NFD', s)
    s = ''.join(ch for ch in s if unicodedata.category(ch) != 'Mn')
    return s.replace('đ', 'd').replace('Đ', 'D').lower().strip()

t0 = time.time()
os.makedirs(LOCAL_DATA, exist_ok=True)
for name in sorted(os.listdir(DRIVE_DATA)):
    src = os.path.join(DRIVE_DATA, name)
    if not os.path.isdir(src):
        continue
    if norm_key(name) not in SUBJECT_MAP:
        print(f'  bỏ qua thư mục: {name}')
        continue
    dst = os.path.join(LOCAL_DATA, name)
    if os.path.isdir(dst):
        print(f'  đã có bản sao: {name}')
        continue
    print(f'  sao chép {name} ...', flush=True)
    shutil.copytree(src, dst)
print(f'Xong sau {time.time()-t0:.0f}s')
!du -sh /content/Dataset2/*

## 4. Cấu hình và hàm trích xuất (giữ nguyên logic của `extract_keypoint.py`)

In [ ]:
import re, json, pickle, hashlib
from pathlib import Path
from tqdm.auto import tqdm

FPS                   = 30
SEQ_LEN               = 60
MIN_FRAMES            = 30
MIN_DETECTED_FRAMES   = 30
MIN_VALID_FRAME_RATIO = 0.5
FALL_STRIDE           = SEQ_LEN     # không chồng lấn
CONF_THRESH           = 0.1
NUM_KEYPOINTS         = 14

FALL_CATEGORIES = ['Lying_Falls', 'Sitting_Falls', 'Standing_Falls']

mp_pose = mp.solutions.pose
KEYPOINT_ORDER = [
    'Nose', 'Left Shoulder', 'Right Shoulder', 'Left Elbow', 'Right Elbow',
    'Left Wrist', 'Right Wrist', 'Left Hip', 'Right Hip', 'Left Knee',
    'Right Knee', 'Left Ankle', 'Right Ankle',
    # 13: Mid-Hip (ảo) = (Left Hip + Right Hip) / 2
]
LANDMARK_MAP = {n: getattr(mp_pose.PoseLandmark, n.upper().replace(' ', '_'))
                for n in KEYPOINT_ORDER}
PARENT_MAP = {13: None, 7: 13, 8: 13, 1: 7, 2: 8, 0: 1, 3: 1, 4: 2,
              5: 3, 6: 4, 9: 7, 10: 8, 11: 9, 12: 10}


def extract_keypoints_single(landmarks, img_w, img_h):
    kps = []
    for name in KEYPOINT_ORDER:
        try:
            lm = landmarks.landmark[LANDMARK_MAP[name].value]
            kps.append((lm.x * img_w, lm.y * img_h,
                        getattr(lm, 'visibility', getattr(lm, 'presence', 0.0))))
        except Exception:
            return None
    arr = np.array(kps, dtype=np.float32)
    mid_xy = (arr[7, :2] + arr[8, :2]) / 2.0
    mid_conf = np.minimum(arr[7, 2], arr[8, 2])
    return np.vstack([arr, np.array([mid_xy[0], mid_xy[1], mid_conf], dtype=np.float32)])


def compute_motion(j):
    m = np.zeros_like(j); m[1:] = j[1:] - j[:-1]; m[:, :, 2] = j[:, :, 2]; return m


def compute_bone_angle(j):
    F, V, _ = j.shape
    angle = np.zeros((F, V, 3), dtype=np.float32)
    for v in range(V):
        p = PARENT_MAP[v]
        if p is None:
            angle[:, v, 0] = 1.0; angle[:, v, 2] = j[:, v, 2]
        else:
            th = np.arctan2(j[:, v, 1] - j[:, p, 1], j[:, v, 0] - j[:, p, 0])
            angle[:, v, 0] = np.cos(th); angle[:, v, 1] = np.sin(th)
            angle[:, v, 2] = np.minimum(j[:, v, 2], j[:, p, 2])
    return angle


def interpolate_missing_keypoint_frames(raw_kps):
    if not raw_kps:
        return None, None, 0
    F = len(raw_kps)
    arr = np.full((F, NUM_KEYPOINTS, 3), np.nan, dtype=np.float32)
    valid = np.zeros(F, dtype=bool)
    for i, kp in enumerate(raw_kps):
        if kp is not None:
            arr[i] = kp; valid[i] = True
    n_valid = int(valid.sum())
    if n_valid == 0:
        return None, None, 0
    idx = np.arange(F); vidx = idx[valid]
    filled = np.empty_like(arr)
    for v in range(NUM_KEYPOINTS):
        for c in range(3):
            filled[:, v, c] = np.interp(idx, vidx, arr[valid, v, c])
    return filled.astype(np.float32), valid, n_valid


def extract_raw_from_video(video_path):
    # Trả về dict: skel (F,14,3) đã nội suy hoặc None, mask, img_w, img_h, n_frames, n_detected
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return dict(skel=None, mask=None, img_w=0, img_h=0, n_frames=0, n_detected=0, err='cannot_open')
    img_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); img_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    raw, first_shape = [], None
    with mp_pose.Pose(static_image_mode=False, model_complexity=1, smooth_landmarks=False,
                      min_detection_confidence=CONF_THRESH,
                      min_tracking_confidence=CONF_THRESH) as pose:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if first_shape is None:
                first_shape = frame.shape[:2]   # (h, w) thực tế sau khi OpenCV giải mã
            res = pose.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            raw.append(extract_keypoints_single(res.pose_landmarks, img_w, img_h)
                       if res.pose_landmarks else None)
    cap.release()
    # Nếu OpenCV tự xoay khung hình theo metadata, kích thước frame thật khác CAP_PROP -> cảnh báo
    shape_mismatch = first_shape is not None and (first_shape[1], first_shape[0]) != (img_w, img_h)
    out = dict(img_w=img_w, img_h=img_h, n_frames=len(raw),
               n_detected=int(sum(k is not None for k in raw)),
               shape_mismatch=bool(shape_mismatch), err='')
    if len(raw) < MIN_FRAMES:
        out.update(skel=None, mask=None, err='too_few_frames'); return out
    skel, mask, n_valid = interpolate_missing_keypoint_frames(raw)
    if skel is None or n_valid < MIN_DETECTED_FRAMES:
        out.update(skel=None, mask=None, err='too_few_detected'); return out
    out.update(skel=skel, mask=mask)
    return out


def _pad(arr):
    F = arr.shape[0]
    if F >= SEQ_LEN:
        return arr[:SEQ_LEN]
    return np.concatenate([arr, np.repeat(arr[-1:], SEQ_LEN - F, axis=0)])


def make_windows(skel, mask):
    # Giống make_fall_windows / make_nonfall_windows cũ (hai hàm cũ cho kết quả như nhau vì FALL_STRIDE = SEQ_LEN)
    F = len(skel); wins = []
    if F < MIN_FRAMES:
        return wins
    if F < SEQ_LEN:
        if _pad(mask).mean() >= MIN_VALID_FRAME_RATIO:
            wins.append((0, _pad(skel)))
        return wins
    p = 0
    while p + SEQ_LEN <= F:
        if mask[p:p + SEQ_LEN].mean() >= MIN_VALID_FRAME_RATIO:
            wins.append((p, skel[p:p + SEQ_LEN].copy()))
        p += FALL_STRIDE
    return wins

## 5. Liệt kê video và gán metadata

Mỗi người đặt tên thư mục một kiểu, nên nhãn được suy ra từ đường dẫn đã chuẩn hóa:

| Đường dẫn chứa | category | nonfall_type |
|---|---|---|
| `lying` | Lying_Falls | – |
| `sitting` | Sitting_Falls | – |
| `standing` | Standing_Falls | – |
| `de nham voi nga` | non_fall | hard |
| `thuong ngay` | non_fall | adl |
| `non_fall` (Minh, Toàn) | non_fall | unspecified |

`view` lấy từ `cam1`/`cam2` trong tên file. `group_id` = người + tên file bỏ phần cam, để hai góc quay
của cùng một lần thực hiện (nếu tên trùng) luôn nằm cùng phía khi chia train/val.

In [ ]:
import pandas as pd

def infer_category(rel_parts):
    k = norm_key('/'.join(rel_parts[:-1]))
    if 'lying' in k:    return 'Lying_Falls', 1, ''
    if 'sitting' in k:  return 'Sitting_Falls', 1, ''
    if 'standing' in k: return 'Standing_Falls', 1, ''
    if 'de nham' in k:  return 'non_fall', 0, 'hard'
    if 'thuong ngay' in k: return 'non_fall', 0, 'adl'
    if 'non' in k:      return 'non_fall', 0, 'unspecified'
    return None, None, None

videos = []
for subj_dir in sorted(Path(LOCAL_DATA).iterdir()):
    skey = norm_key(subj_dir.name)
    if not subj_dir.is_dir() or skey not in SUBJECT_MAP:
        continue
    for vf in sorted(subj_dir.rglob('*')):
        if vf.suffix.lower() not in ('.mp4', '.mov', '.avi'):
            continue
        rel = vf.relative_to(subj_dir).parts
        cat, label, nf_type = infer_category(rel)
        if cat is None:
            print('  [WARN] không suy ra được nhãn:', vf); continue
        m = re.search(r'cam\s*(\d)', vf.stem, re.I)
        view = f'cam{m.group(1)}' if m else 'unknown'
        stem_nocam = re.sub(r'_?cam\s*\d', '', vf.stem, flags=re.I)
        act = re.match(r'^\d+_[A-Za-z]+_([A-Za-z]+)_', vf.stem)
        rel_path = unicodedata.normalize('NFC', str(vf.relative_to(LOCAL_DATA)))
        videos.append(dict(
            path=str(vf), rel_path=rel_path,
            subject=SUBJECT_MAP[skey], view=view, category=cat, label=label,
            nonfall_type=nf_type, activity=act.group(1) if act else '',
            group_id=f'{SUBJECT_MAP[skey]}/{norm_key(stem_nocam)}',
            cache_key=hashlib.md5(rel_path.encode('utf-8')).hexdigest(),
        ))

vdf = pd.DataFrame(videos)
vdf['video_id'] = np.arange(len(vdf))
print('Tổng số video:', len(vdf))
display(pd.crosstab([vdf.subject, vdf.category], vdf.view, margins=True))
display(pd.crosstab(vdf.subject, vdf.nonfall_type.replace('', '-')))
assert (vdf.view != 'unknown').all(), 'Có video không có cam1/cam2 trong tên'

## 6. Trích xuất khung xương (song song, có cache trên Drive)

Mỗi video được lưu một file `.pkl` trong `kp_cache_v2/`. Chạy lại cell này sẽ bỏ qua video đã có cache.
Nếu gặp lỗi khi chạy song song, đặt `N_WORKERS = 1`.

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed

N_WORKERS = max(1, os.cpu_count() or 1)

def _work(args):
    path, cache_path = args
    try:
        out = extract_raw_from_video(path)
    except Exception as e:
        out = dict(skel=None, mask=None, img_w=0, img_h=0, n_frames=0, n_detected=0,
                   shape_mismatch=False, err=f'exception: {e}')
    with open(cache_path, 'wb') as f:
        pickle.dump(out, f)
    return path

todo = [(r.path, os.path.join(CACHE_DIR, r.cache_key + '.pkl'))
        for r in vdf.itertuples() if not os.path.exists(os.path.join(CACHE_DIR, r.cache_key + '.pkl'))]
print(f'Cần trích: {len(todo)} / {len(vdf)} video  |  workers={N_WORKERS}')

t0 = time.time()
if N_WORKERS == 1:
    for a in tqdm(todo): _work(a)
else:
    with ProcessPoolExecutor(N_WORKERS) as ex:
        futs = [ex.submit(_work, a) for a in todo]
        for _ in tqdm(as_completed(futs), total=len(futs)): pass
print(f'Xong sau {(time.time()-t0)/60:.1f} phút')

## 7. Cắt cửa sổ, tạo `custom_all_v2.npz` và `videos_v2.csv`

In [ ]:
rows, J, M, A = [], [], [], []
meta = {k: [] for k in ['y', 'img_w', 'img_h', 'seq_id', 'video_id', 'window_idx', 'subject', 'view',
                        'category', 'nonfall_type', 'activity', 'group_id', 'orientation', 'is_flip']}
for r in tqdm(vdf.itertuples(), total=len(vdf)):
    with open(os.path.join(CACHE_DIR, r.cache_key + '.pkl'), 'rb') as f:
        out = pickle.load(f)
    wins = make_windows(out['skel'], out['mask']) if out['skel'] is not None else []
    orient = 'portrait' if out['img_h'] > out['img_w'] else 'landscape'
    rows.append(dict(video_id=r.video_id, rel_path=r.rel_path, subject=r.subject, view=r.view,
                     category=r.category, label=r.label, nonfall_type=r.nonfall_type,
                     activity=r.activity, group_id=r.group_id, img_w=out['img_w'], img_h=out['img_h'],
                     orientation=orient, n_frames=out['n_frames'], n_detected=out['n_detected'],
                     detected_ratio=out['n_detected'] / max(1, out['n_frames']),
                     shape_mismatch=out.get('shape_mismatch', False),
                     n_windows=len(wins), error=out['err'] or ('no_valid_window' if not wins else '')))
    for wi, (start, w) in enumerate(wins):
        J.append(w); M.append(compute_motion(w)); A.append(compute_bone_angle(w))
        for k, v in dict(y=r.label, img_w=out['img_w'], img_h=out['img_h'], seq_id=r.video_id,
                         video_id=r.video_id, window_idx=wi, subject=r.subject, view=r.view,
                         category=r.category, nonfall_type=r.nonfall_type, activity=r.activity,
                         group_id=r.group_id, orientation=orient, is_flip=0).items():
            meta[k].append(v)

stats = pd.DataFrame(rows)
stats.to_csv(OUT_CSV, index=False, encoding='utf-8-sig')

str_keys = ['subject', 'view', 'category', 'nonfall_type', 'activity', 'group_id', 'orientation']
np.savez_compressed(
    OUT_NPZ,
    X_joint=np.asarray(J, np.float32), X_motion=np.asarray(M, np.float32), X_angle=np.asarray(A, np.float32),
    **{k: np.asarray(v, dtype='U64' if k in str_keys else np.int64 if k == 'y' else np.int32)
       for k, v in meta.items()},
)
print(f'-> {OUT_NPZ}  windows={len(J)}  videos dùng được={int((stats.n_windows > 0).sum())}/{len(stats)}')
print(f'-> {OUT_CSV}')

## 8. Kiểm tra (sanity check)

Bảng dưới là số liệu cho mục mô tả dataset (R2.1: số người, số video, số cửa sổ). Đặc biệt chú ý:

* **Video bị loại** (lỗi / không có cửa sổ hợp lệ).
* **`shape_mismatch`**: phải bằng 0. Nếu khác 0 nghĩa là OpenCV tự xoay khung hình, tọa độ pixel sẽ sai trục.
* **Hướng video × nhãn**: nếu hướng dọc/ngang gắn với nhãn trong một người, mô hình có thể học lối tắt.

In [ ]:
print('Video bị loại:'); display(stats[stats.error != ''][['rel_path', 'n_frames', 'n_detected', 'error']])
print('shape_mismatch =', int(stats.shape_mismatch.sum()))
print('\nSố video / số cửa sổ theo người × nhãn × cam:')
display(stats.pivot_table(index=['subject', 'category'], columns='view',
                          values=['video_id', 'n_windows'], aggfunc={'video_id': 'count', 'n_windows': 'sum'},
                          margins=True))
print('\nHướng video × nhãn theo người:')
display(pd.crosstab([stats.subject, stats.orientation], stats.label))
print('\nTỷ lệ frame phát hiện được khung xương:')
display(stats.groupby(['subject', 'view']).detected_ratio.describe()[['mean', 'min', '25%']])

In [ ]:
# Vẽ khung xương lên 1 frame của 1 video dọc và 1 video ngang để kiểm tra tọa độ khớp đúng chỗ
import matplotlib.pyplot as plt
EDGES = [(0,1),(0,2),(1,2),(1,3),(3,5),(2,4),(4,6),(1,7),(2,8),(7,8),(7,9),(9,11),(8,10),(10,12),(7,13),(8,13)]
picks = []
for o in ['portrait', 'landscape']:
    s = stats[(stats.orientation == o) & (stats.n_windows > 0) & (stats.label == 1)]
    if len(s): picks.append(s.iloc[0])
fig, axes = plt.subplots(1, len(picks), figsize=(6 * len(picks), 7))
axes = np.atleast_1d(axes)
for ax, r in zip(axes, picks):
    cache = pickle.load(open(os.path.join(CACHE_DIR, vdf.loc[r.video_id, 'cache_key'] + '.pkl'), 'rb'))
    fi = min(30, len(cache['skel']) - 1)
    cap = cv2.VideoCapture(vdf.loc[r.video_id, 'path']); cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
    ok, frame = cap.read(); cap.release()
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); k = cache['skel'][fi]
    for a, b in EDGES: ax.plot([k[a, 0], k[b, 0]], [k[a, 1], k[b, 1]], 'lime', lw=2)
    ax.scatter(k[:, 0], k[:, 1], c='red', s=15); ax.set_title(f'{r.orientation} {r.img_w}x{r.img_h}'); ax.axis('off')
plt.tight_layout(); plt.savefig(os.path.join(SANITY_DIR, 'skeleton_overlay.png'), dpi=80); plt.show()